# AI Try-On Backend - Google Colab Setup

This notebook sets up the try-on backend project in Google Colab for ML model development and testing.

## Features
- Install all dependencies
- Clone/download project from GitHub
- Setup ML models with GPU support
- Test ML inference
- Optional: Run API server with ngrok


## Step 1: Install Dependencies

Install all required packages for ML models and backend.


In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-glx libglib2.0-0

# Install PyTorch with CUDA (for GPU support)
!pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118

# Install other dependencies
!pip install -q numpy==1.24.3 opencv-python==4.8.1.78 Pillow==10.1.0 scipy==1.11.4
!pip install -q imageio==2.31.5 scikit-image==0.22.0
!pip install -q fastapi==0.104.1 uvicorn[standard]==0.24.0 python-multipart==0.0.6
!pip install -q pydantic==2.5.0 pydantic-settings==2.1.0
!pip install -q requests==2.31.0 tqdm==4.66.1
!pip install -q pyngrok  # For exposing API to public

print("✅ All dependencies installed!")


## Step 2: Clone Project

Clone the repository from GitHub.


In [ ]:
import os
import sys
from pathlib import Path

# Clone repository
repo_url = "https://github.com/StefanusSimandjuntak111/try-on-backend.git"
project_dir = "/content/try-on-backend"

if os.path.exists(project_dir):
    print(f"📁 Project already exists at {project_dir}")
    print("🔄 Updating repository...")
    !cd {project_dir} && git pull
else:
    print(f"📥 Cloning repository to {project_dir}...")
    !git clone {repo_url} {project_dir}

# Add to Python path
sys.path.insert(0, project_dir)
os.chdir(project_dir)

print(f"✅ Project cloned to {project_dir}")
print(f"📂 Current directory: {os.getcwd()}")


## Step 3: Setup Environment

Configure environment variables for Colab.


In [ ]:
import torch

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Create weights directory
weights_dir = Path("weights")
weights_dir.mkdir(exist_ok=True)
print(f"📁 Weights directory: {weights_dir.absolute()}")

# Create uploads directory
uploads_dir = Path("uploads")
uploads_dir.mkdir(exist_ok=True)

# Create results directory
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

print("✅ Environment setup complete!")


## Step 4: Test ML Models

Test loading and using ML models.


In [ ]:
from app.config import settings

# Update settings for Colab
settings.ML_DEVICE = device

print("🧪 Testing ML models...")

# Test U2-Net (Background Removal)
try:
    from app.ml.background.model import get_u2net_model
    u2net = get_u2net_model()
    print(f"✅ U2-Net loaded (device: {u2net.device})")
except Exception as e:
    print(f"⚠️  U2-Net: {str(e)}")

# Test SCHP (Human Parsing)
try:
    from app.ml.parsing.model import get_schp_model
    schp = get_schp_model()
    print(f"✅ SCHP loaded (device: {schp.device})")
except Exception as e:
    print(f"⚠️  SCHP: {str(e)}")

# Test OpenPose
try:
    from app.ml.pose.model import get_openpose_model
    openpose = get_openpose_model()
    print(f"✅ OpenPose loaded (device: {openpose.device})")
except Exception as e:
    print(f"⚠️  OpenPose: {str(e)}")

# Test HR-VITON
try:
    from app.ml.hrviton.model import get_hrviton_model
    hrviton = get_hrviton_model()
    print(f"✅ HR-VITON loaded (device: {hrviton.device})")
except Exception as e:
    print(f"⚠️  HR-VITON: {str(e)}")

print("\n✅ ML model testing complete!")


## Step 5: Run API Server (Optional)

Run the FastAPI server and expose it via ngrok.


In [ ]:
# Setup ngrok (get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken)
from pyngrok import ngrok

# Replace with your ngrok authtoken (optional for free tier)
NGROK_AUTHTOKEN = ""  # Set your token here if you have one
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"🌐 Public URL: {public_url}")
print(f"📚 API Docs: {public_url}/docs")


In [ ]:
# Start FastAPI server
import subprocess
import threading

def run_server():
    # Set environment variables
    import os
    os.environ["ML_DEVICE"] = device
    os.environ["DATABASE_URL"] = "sqlite:///./test.db"  # Use SQLite for Colab
    
    # Run server
    subprocess.run([
        "python", "-m", "uvicorn",
        "app.main:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ])

# Run server in background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("🚀 Server starting...")
print("⏳ Wait a few seconds for server to start")
print(f"📡 Server should be available at: {public_url if 'public_url' in locals() else 'http://localhost:8000'}")
